# Phase 4b - Scaled self-play long-run (the real GO/NO-GO)

What the toy probes settled (see `rl_research/PROBE_ARCHALUDON_2026-06-30.md`):
- **Stability: solved.** The orbit-wars anti-collapse recipe (LR 1e-4, batch 256, 1 epoch,
  entropy floor 0.04) trained `medium` (15.8M) to completion with healthy entropy - no collapse.
- **The in-loop eval was lying.** `quick_eval` silently degraded `kaggle:`/`mirror` opponents
  to the trivial 'first' baseline, so the "92-95%" curves were meaningless. **Now fixed** -
  `quick_eval` runs the real `kaggle:<name>` agent on its own deck (or raises). Re-`git pull`.
- **The real numbers (scripts/eval.py, n=160 vs held-out `kaggle:archaludon`):** small 25.6%,
  medium 28.7% - i.e. both **lose** to the strong matched rule agent, and **capacity barely moved
  it** (25.6 -> 28.7, CIs overlap). The bottleneck is **steps, not params**: we trained ~3M
  decisions; Gerar hit top-2% at 412M (~130x), the big runs at 10-15B. We are badly undertrained.

**So this run is the actual go/no-go: scale the step budget ~6-8x on the stable recipe and read
the REAL `kaggle:archaludon` slope** (now that the in-loop eval is honest). Deck-controlled as
before: Archaludon trainee on the rule agent's exact 60-card list, `kaggle:archaludon` held OUT
of the training pool and used only as the eval yardstick.

**Read `eval: ... kaggle:archaludon`** every 25 iters (this is now the true number):
- climbing through ~40-50% and still rising -> **GO**, fan out A+B at full budget.
- flat in the 25-35% band it started at -> steps alone won't close it -> **BC warm-start**
  (see `orbit-wars-selfplay-lessons` + the BC memory note) is the next lever.

Note: no `--resume` yet, so treat this as one long session (~12-16h at ~40s/iter). Checkpoints
persist to Drive; if it dies, restart loses optimizer/iter progress. Bump `ITERS` to fit budget.

In [ ]:
# Mount Drive (artifact persistence), then clone (or update) the repo and cd in.
import os

from google.colab import drive
drive.mount('/content/drive')

# Everything the runs produce is written under here so it survives the session and
# can be pulled back with scripts/download_artifacts.py. Keep this name in sync with
# DRIVE_BASE in that script ('ptcg_outputs').
DRIVE_OUTPUT = '/content/drive/MyDrive/ptcg_outputs'
for sub in ['', '/logs', '/checkpoints_sp_small', '/ablation_sp']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = "https://github.com/oshbocker/pokemon_tcg_ai_battle.git"
REPO_DIR = "/content/pokemon_tcg_ai_battle"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

assert os.path.isfile("agent/cg/libcg.so"), (
    "agent/cg/libcg.so missing - push it to the repo (see .gitignore exception)."
)
import torch
print("repo:", os.getcwd(), "| Drive:", DRIVE_OUTPUT)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# Hardware topology - record this alongside the results.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "(no GPU runtime selected)"
!nproc
!lscpu | grep -E "^CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Model name"
!free -h | head -2


## Scaled long-run (`medium`, stable recipe, honest in-loop eval)

Same recipe that trained `medium_v2` cleanly, scaled on **iters** (the identified lever).
`--eval-opponents random,heuristic,kaggle:archaludon` now actually runs the hand-coded
Archaludon (the `quick_eval` fix); `mirror` is dropped (it now raises - use the gate signal).
Watch `ent` stays > ~0.05 and the `kaggle:archaludon` eval slope.

**Gate = `--gate-vs pool`** (2026-06-30): `best.pt` is now crowned by the current net's
**weighted win-rate across the whole training league** (self + past checkpoints + the manifest
agents: Starmie/Dragapult/romanrozen/heuristic/random), not just a mirror vs the frozen best.
Watch the per-opponent breakdown in the `gate pool NN% [self .. | starmie .. | ..]` line - it
is a much better proxy for ladder strength than the mirror number, which the old gate could
maximise while staying blind to the aggro decks we meet on the ladder. The gate is **relative**
(promote when the pool score beats the last-promoted checkpoint's), with `--gate-threshold` as a
light absolute floor. `kaggle:archaludon` stays held OUT of the manifest, so the eval column is
still an honest yardstick the gate never optimises against.

In [ ]:
# -- Scaled self-play long-run: the real go/no-go --
ITERS = 1200   # ~6x medium_v2's 200; bump/cut to fit the session budget (~40s/iter)
PROBE_OUT = f'{DRIVE_OUTPUT}/probe_archaludon_medium_long'
LOG = '/content/colab_probe_archaludon_medium_long.txt'
# PFSP (conservative first pass): after a 200-iter static warmup, re-weight the manifest
# opponents by the learner's live win-rate. `var` mode focuses games on contested (~50%)
# matchups. Modest bounds keep it gentle and never starve a pool agent: each opponent's
# weight stays within [0.5x, 2x] its manifest base (--pfsp-wmin 0.5 = the floor). self/past
# are untouched. Watch the new `| mix:` log line (realized share) + any `FORFEIT=` alarm.
!python scripts/train_selfplay.py \
    --deck agent/kaggle_agents/archaludon_deck.csv \
    --opp-manifest agent/opponents/mixed_pool_heldout_archaludon.json \
    --size medium --collector dist --workers 48 \
    --opponent self --league --pool-size 5 --snapshot-every 25 \
    --iters {ITERS} --games-per-iter 256 --epochs 1 --minibatch 256 \
    --lr 1e-4 --lr-final 1e-5 --ent-coef 0.04 --target-entropy 0.15 --target-kl 1.5 \
    --gate --gate-vs pool --gate-every 25 --gate-games 120 --gate-past 2 --gate-threshold 0.4 \
    --eval-every 25 --eval-games 120 \
    --eval-opponents random,heuristic,kaggle:archaludon \
    --pfsp --pfsp-mode var --pfsp-after 200 --pfsp-wmin 0.5 --pfsp-wmax 2.0 --pfsp-ema 0.8 \
    --device cuda --seed 0 \
    --out {PROBE_OUT} 2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist log to Drive


## Alakazam-hardened warm-start resume (fix the 15% Alakazam matchup)

The [2026-07-01 replay audit](../rl_research/STRATEGY_WRITEUP_LOG.md) found the shipped
Archaludon (LB 1047) goes **2-11 (15%)** vs Alakazam combo — the ladder's #1 deck — even
though it trained against the heuristic `kaggle:alakazam`. The rule agent is simply far
weaker than the real Alakazam pilots it meets on the ladder. This run **resumes from that
`best.pt`** (`--init-ckpt`) and hardens it against *strong* Alakazam:

- **v2 manifest** (`mixed_pool_archaludon_v2_alakazam.json`) bumps the heuristic
  `kaggle:alakazam` exploiter 1.5 → 2.0.
- The **trained Alakazam RL agent** (`selfplay_alakazam_medium/best.pt`, our LB-913 B agent)
  is added as a `--league-checkpoint` @2.0, piloting its own deck — a much tougher Alakazam
  sparring partner. Combined Alakazam pressure (~4.0) is the dominant fixed bucket, by design.
- `kaggle:archaludon` stays **held out** as the honest eval yardstick; `kaggle:alakazam` is
  added to `--eval-opponents` so the matchup slope is readable directly.
- Warm continuation, not a cold run: **LR 5e-5 → 5e-6, 500 iters**, PFSP after just 50 iters
  (the net is already competent). `--init-ckpt` also seeds a never-evicted `pool['parent']`
  anchor so the run cannot silently regress below the LB-1047 checkpoint.

**Success = the gate-pool `alakazam_rl` bucket + eval `kaggle:alakazam` win-rate climb out of
the teens toward parity**, without the other buckets (starmie/dragapult) collapsing. Re-run the
`scout-agents` skill after submitting to confirm the ladder 15% actually closes.

> Both checkpoints must already be on Drive under `ptcg_outputs/` — they are produced by the
> Archaludon go/no-go cell above and the Alakazam notebook respectively. If you pruned Drive,
> re-upload them (or `download_artifacts` → re-sync) before running.

In [ ]:
# -- Alakazam-hardened WARM-START resume: fix Archaludon's 15% Alakazam matchup --
# Resumes from the shipped Archaludon best.pt (it850, LB 1047) and hardens it against
# STRONG Alakazam. The shipped agent went 2-11 (15%) vs Alakazam combo on the ladder
# despite training vs the heuristic kaggle:alakazam -- the rule agent is far weaker than
# real Alakazam pilots. Two changes bring real Alakazam pressure:
#   (a) v2 manifest bumps heuristic kaggle:alakazam 1.5 -> 2.0;
#   (b) the TRAINED Alakazam RL best.pt is added as a --league-checkpoint @2.0 piloting
#       its own deck (combined Alakazam pressure ~4.0 = the dominant fixed bucket, by design).
RESUME_ITERS = 500                                                # warm continuation, not a cold 1200 run
PARENT      = f'{DRIVE_OUTPUT}/probe_archaludon_medium_long/best.pt'  # previous Archaludon best (it850, LB 1047)
ALAKAZAM_RL = f'{DRIVE_OUTPUT}/selfplay_alakazam_medium/best.pt'      # best Alakazam RL agent (it500, LB 913)
OUT = f'{DRIVE_OUTPUT}/archaludon_alakazam_hardened'
LOG = '/content/colab_archaludon_alakazam_hardened.txt'
import os
assert os.path.isfile(PARENT),      f'missing parent ckpt on Drive: {PARENT}'
assert os.path.isfile(ALAKAZAM_RL), f'missing Alakazam RL ckpt on Drive: {ALAKAZAM_RL}'
# --init-ckpt warm-starts the trainee from PARENT (cfg=medium is read from the checkpoint,
# so --size is only a log label) and seeds BOTH the gate's frozen_best AND a never-evicted
# pool['parent'] anchor -> the run cannot silently regress below the LB-1047 checkpoint.
# LR is lower than the cold run (5e-5 -> 5e-6): re-adapt to the harder pool, don't overwrite.
# PFSP activates early (--pfsp-after 50) since the net is already competent. kaggle:archaludon
# stays HELD OUT (honest eval yardstick); kaggle:alakazam is added to --eval-opponents so we
# can read the Alakazam matchup slope directly.
!python scripts/train_selfplay.py \
    --init-ckpt {PARENT} \
    --size medium --deck agent/kaggle_agents/archaludon_deck.csv \
    --opp-manifest agent/opponents/mixed_pool_archaludon_v2_alakazam.json \
    --league-checkpoint {ALAKAZAM_RL}:2.0:agent/kaggle_agents/alakazam_deck.csv:alakazam_rl \
    --collector dist --workers 48 \
    --opponent self --league --pool-size 5 --snapshot-every 25 \
    --iters {RESUME_ITERS} --games-per-iter 256 --epochs 1 --minibatch 256 \
    --lr 5e-5 --lr-final 5e-6 --ent-coef 0.04 --target-entropy 0.15 --target-kl 1.5 \
    --gate --gate-vs pool --gate-every 25 --gate-games 120 --gate-past 2 --gate-threshold 0.4 \
    --eval-every 25 --eval-games 120 \
    --eval-opponents random,heuristic,kaggle:archaludon,kaggle:alakazam \
    --pfsp --pfsp-mode var --pfsp-after 50 --pfsp-wmin 0.5 --pfsp-wmax 2.0 --pfsp-ema 0.8 \
    --device cuda --seed 0 \
    --out {OUT} 2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist log to Drive


## Meta-ON lineage-root re-train (`use_card_meta`) — the definitive coevolution parent

The [2026-07-01 novel-card kill-criterion](../rl_research/STRATEGY_WRITEUP_LOG.md) PASSED:
grafting the frozen card-metadata feature at fine-tune time rescued a novel-card deck
mutation (Genesect swap: zero-shot 71.9% / warm-OFF 74.8% / **warm-ON 79.7%** vs ref 80.4%).
The remaining confound: the metadata projection there was **zero-init and learned during the
30-iter fine-tune itself**. The definitive setup is a parent whose metadata pathway is
already *mature*, so a search mutation only has to slot a novel card into an existing
representation. This run produces that parent — and makes it the **new lineage root**:

- **Root = hardened-it400** (`archaludon_alakazam_hardened/best.pt`, gate-pool 69.4%), our
  best Archaludon — not the older LB-1047 `probe_archaludon_medium_long` checkpoint, so the
  metadata pathway matures on the lineage we actually carry forward.
- **Warm continuation, not a cold 1200-iter run**: `--init-ckpt` + `--use-card-meta`. The
  graft is behavior-identical at iter 0 (zero-init, bias-free projection; check the load
  report prints `card_meta=True` and `missing=['card_meta_table', 'card_meta_proj.weight']`).
- **Ladder-variant sparring (new vs the hardened run).** The [2026-07-02 scout](../rl_research/STRATEGY_WRITEUP_LOG.md)
  found the it400 sub goes **3-6 (33%) in the Archaludon pseudo-mirror**, 0-2 vs lists that
  swap in **Judge** — hand disruption it has never seen in training, and its losses there are
  0-prize setup collapses. The exact observed lists (`agent/decks/archaludon_ladder_judge.csv`
  @1.5, `archaludon_ladder_xerosic.csv` @0.75; engine-validated) join the league as
  `--league-checkpoint`s **piloted by the root checkpoint itself** — a strong near-mirror
  pilot with disruption tech. Watch the `arch_judge` / `arch_xerosic` gate buckets climb.
- **Everything else mirrors the hardened run it continues from** (original deck, v2 alakazam
  manifest + `alakazam_rl` league-checkpoint @2.0, same gate/PFSP). `kaggle:archaludon`
  stays held out as the honest yardstick.
- **CSV gotcha (handled in the cell, do not skip it):** `data/EN_Card_Data.csv` is
  gitignored, so the Colab clone doesn't have it — and without it the model silently builds
  a **zero** metadata table (dead feature, invalid run). The cell copies it from
  `ptcg_outputs/data/` on Drive (already uploaded via rclone) and hard-asserts it's real.

**Afterwards** (back on the laptop): pull the checkpoint
(`download_artifacts.py --run archaludon_hardened_meta`), point `PARENT_CKPT` in
`scripts/run_coevo_meta_kill.sh` at it, and rebuild the ref eval (parent-on-original) since
the parent changed — then re-run the novel-card 2×2 for the deconfounded answer, and use
this checkpoint as the warm-start parent for all coevolution search children.

**Coevo note:** the three ladder swaps (`archaludon_ladder_{judge,judge_metal,xerosic}.csv`)
double as **seeded mutation candidates** for the deck search. All three swap **Trainers**, and
card-metadata is blind to Trainer effect text — so they must route straight to the warm-start
fine-tune stage, **never** the zero-shot pre-filter (see COEVOLUTIONARY_DECK_SEARCH.md).

In [ ]:
# -- Meta-ON lineage-root re-train: graft + mature the frozen card-metadata pathway --
# Warm-starts hardened-it400 (the new best Archaludon, gate-pool 69.4%) with
# --use-card-meta and matures the (zero-init) metadata projection, producing the new
# LINEAGE ROOT for the coevolution deck search. League/deck/recipe mirror the
# alakazam-hardened run it continues from; the deltas vs that parent are the metadata
# feature (+ extra steps) and the two LADDER-VARIANT sparring partners below.
#
# LADDER-VARIANT SPARRING (2026-07-02 scout finding): the it400 sub goes 3-6 (33%) in
# the Archaludon pseudo-mirror, 0-2 vs lists that swap in Judge — hand disruption the
# agent has NEVER seen in training (its losses there are 0-prize setup collapses). The
# exact observed lists (agent/decks/archaludon_ladder_*.csv, engine-validated) are added
# as league opponents piloted by the FIXED root checkpoint (not PARENT, so sparring
# stays constant across crash-resumes): judge @1.5, xerosic @0.75.
#
# AUTO-RESUME: Colab kills these runs (silent OOM of the trainer / runtime death), so
# this cell is safe to just re-run after a crash. It warm-starts from whichever of
# OUT/{last,best}.pt got furthest (train_selfplay now writes last.pt every gate/eval
# interval), computes the remaining iters, and continues the LR schedule from where it
# died. Fresh runtime recommended (the engine leaks RAM; a reused session dies fast).
import os
import shutil
import sys

import torch

# The Drive FUSE mount can die mid-session ("Transport endpoint is not connected",
# seen 2026-07-02 killing a league-ckpt torch.load). Verify it's alive, remount if
# not, and STAGE every frozen input checkpoint to local /content disk so training
# never torch.load's from Drive — only OUT/ writes and the log still touch it.
def _drive_alive():
    try:
        os.listdir(DRIVE_OUTPUT)
        return True
    except OSError:
        return False

if not _drive_alive():
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print('flush_and_unmount failed (expected on a dead mount):', e)
    drive.mount('/content/drive', force_remount=True)
assert _drive_alive(), 'Drive mount is dead - Runtime > Restart session, then rerun'

_STAGE_DIR = '/content/ckpts'
os.makedirs(_STAGE_DIR, exist_ok=True)
def stage(drive_path):
    """Copy a Drive file to local disk (idempotent per run) and return the local path."""
    dst = os.path.join(_STAGE_DIR, drive_path.replace(f'{DRIVE_OUTPUT}/', '').replace('/', '__'))
    shutil.copy(drive_path, dst)
    return dst

# data/EN_Card_Data.csv is gitignored -> absent in the Colab clone. Without it the
# model silently builds a ZERO metadata table (dead feature, invalid run): copy it
# from Drive and hard-assert the built table is real before spending GPU hours.
CARD_CSV = 'data/EN_Card_Data.csv'
if not os.path.isfile(CARD_CSV):
    drive_csv = f'{DRIVE_OUTPUT}/data/EN_Card_Data.csv'
    assert os.path.isfile(drive_csv), (
        f'missing {drive_csv} - from the laptop: '
        'rclone copy data/EN_Card_Data.csv gdrive:ptcg_outputs/data/'
    )
    os.makedirs('data', exist_ok=True)
    shutil.copy(drive_csv, CARD_CSV)
sys.path.insert(0, 'src')
from ptcg_battle.card_meta import build_card_meta_table
_t = build_card_meta_table()
assert _t.sum() > 0, 'metadata table is all zeros - CSV missing or corrupt'
print('card-meta table OK:', _t.shape, 'nonzero rows:', int((_t.sum(axis=1) > 0).sum()))

TOTAL_ITERS = 500                                                       # full run budget
LR_START, LR_FINAL = 5e-5, 5e-6                                         # linear schedule over TOTAL_ITERS
ROOT        = f'{DRIVE_OUTPUT}/archaludon_alakazam_hardened/best.pt'    # hardened it400 = lineage root
ALAKAZAM_RL = f'{DRIVE_OUTPUT}/selfplay_alakazam_medium/best.pt'        # its league sparring partner
JUDGE_DECK   = 'agent/decks/archaludon_ladder_judge.csv'    # our list -1 Boss's Orders +1 Judge (ladder 0-2)
XEROSIC_DECK = 'agent/decks/archaludon_ladder_xerosic.csv'  # our list -Pokegear -Jumbo Ice Cream +2 Xerosic's Machinations
OUT = f'{DRIVE_OUTPUT}/archaludon_hardened_meta'
LOG       = f'{DRIVE_OUTPUT}/logs/colab_archaludon_hardened_meta.txt'   # Drive copy (best effort if the mount flaps)
LOG_LOCAL = '/content/colab_archaludon_hardened_meta.txt'               # local copy always survives a mount flap
assert os.path.isfile(ROOT),        f'missing lineage-root ckpt on Drive: {ROOT}'
assert os.path.isfile(ALAKAZAM_RL), f'missing Alakazam RL ckpt on Drive: {ALAKAZAM_RL}'
assert os.path.isfile(JUDGE_DECK) and os.path.isfile(XEROSIC_DECK), 'ladder-variant decks missing - git pull'

# Resume from whichever checkpoint of THIS run got furthest; else start from the root.
# (A meta-ON ckpt warm-starts with missing=[] - a clean full load; the root, being
# meta-OFF, shows the expected missing=['card_meta_table', 'card_meta_proj.weight'].)
PARENT, done = ROOT, 0
for cand in (f'{OUT}/last.pt', f'{OUT}/best.pt'):
    if os.path.isfile(cand):
        it = int(torch.load(cand, map_location='cpu', weights_only=False).get('iter', 0))
        if it > done:
            PARENT, done = cand, it
ITERS = TOTAL_ITERS - done
assert ITERS > 0, f'run already complete ({done}/{TOTAL_ITERS} iters) - bump TOTAL_ITERS to continue'
frac = done / TOTAL_ITERS
LR0 = LR_START + (LR_FINAL - LR_START) * frac                           # continue the decay, don't restart it
print(f'{"RESUMING" if done else "STARTING"} from {PARENT}  (done={done}, remaining={ITERS}, lr {LR0:.2e}->{LR_FINAL:.0e})')

# Stage all frozen inputs locally (ROOT is loaded 3x: init-or-anchor + 2 variant pilots).
ROOT_L, ALAKAZAM_L, PARENT_L = stage(ROOT), stage(ALAKAZAM_RL), stage(PARENT)

!python scripts/train_selfplay.py \
    --init-ckpt {PARENT_L} --use-card-meta \
    --size medium --deck agent/kaggle_agents/archaludon_deck.csv \
    --opp-manifest agent/opponents/mixed_pool_archaludon_v2_alakazam.json \
    --league-checkpoint {ALAKAZAM_L}:2.0:agent/kaggle_agents/alakazam_deck.csv:alakazam_rl \
    --league-checkpoint {ROOT_L}:1.5:{JUDGE_DECK}:arch_judge \
    --league-checkpoint {ROOT_L}:0.75:{XEROSIC_DECK}:arch_xerosic \
    --collector dist --workers 48 \
    --opponent self --league --pool-size 5 --snapshot-every 25 \
    --iters {ITERS} --games-per-iter 256 --epochs 1 --minibatch 256 \
    --lr {LR0} --lr-final {LR_FINAL} --ent-coef 0.04 --target-entropy 0.15 --target-kl 1.5 \
    --gate --gate-vs pool --gate-every 25 --gate-games 120 --gate-past 2 --gate-threshold 0.4 \
    --eval-every 25 --eval-games 120 \
    --eval-opponents random,heuristic,kaggle:archaludon,kaggle:alakazam \
    --pfsp --pfsp-mode var --pfsp-after 50 --pfsp-wmin 0.5 --pfsp-wmax 2.0 --pfsp-ema 0.8 \
    --device cuda --seed 0 \
    --out {OUT} 2>&1 | tee {LOG_LOCAL} {LOG}
!cp {LOG_LOCAL} {LOG} || echo "(Drive log copy failed - grab {LOG_LOCAL} before the runtime dies)"

## Results are on Drive — pull them to the laptop

Everything the runs produced lives under `MyDrive/ptcg_outputs/` (checkpoints, eval
CSVs, and the `logs/colab_*.txt` records). Nothing needs to be git-pushed from here.
The cell below just confirms what landed; back on the laptop, fetch it with rclone:

```bash
uv run python scripts/download_artifacts.py --logs                 # colab_*.txt -> rl_research/
uv run python scripts/download_artifacts.py --run ablation_sp      # A/B ckpts + CSVs -> outputs/
uv run python scripts/download_artifacts.py --run checkpoints_sp_small
```

Then paste the A/B table + RECOMMENDATION into `ABLATION_OPTION_RANK.md` and commit
the logs. (One-time rclone `gdrive` remote setup is in CLAUDE.md.)

In [ ]:
# Confirm what persisted to Drive.
!ls -lhR {DRIVE_OUTPUT} | sed -n '1,80p'
